## Disponibilizar informacion en la coapa Golden para que sea explotada por el usuario final

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("catalogo", "catalog_movie_project")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
from pyspark.sql.functions import *

### Aagregados por género

In [0]:
query = f"""

CREATE OR REPLACE VIEW {catalogo}.golden.view_sl_kpi_genre AS
SELECT
    genre,
    COUNT(*) AS total_movies,
    ROUND(AVG(user_score), 2) AS avg_rating,
    ROUND(AVG(revenue_usd), 2) AS avg_revenue,
    ROUND(AVG(budget_usd), 2) AS avg_budget,
    ROUND(AVG(roi), 2) AS avg_roi,
    MAX(user_score) AS max_rating
FROM (
    SELECT
        id,
        EXPLODE(genres) AS genre,
        user_score,
        revenue_usd,
        budget_usd,
        roi
    FROM {catalogo}.golden.movie_master
) t
GROUP BY genre
"""
spark.sql(query)

### tendencias temporales

In [0]:
query1 = f"""
CREATE OR REPLACE VIEW {catalogo}.golden.view_sl_kpi_time AS
SELECT
    release_year,
    COUNT(*) AS total_movies,
    ROUND(AVG(user_score), 2) AS avg_rating,
    SUM(revenue_usd) AS total_revenue,
    ROUND(AVG(roi), 2) AS avg_roi
FROM {catalogo}.golden.movie_master
GROUP BY release_year
ORDER BY release_year
"""
spark.sql(query1)

### ranking + clasificación

In [0]:
query2 = f"""
CREATE OR REPLACE VIEW {catalogo}.golden.view_sl_profitability AS
SELECT
    id,
    title,
    budget_usd,
    revenue_usd,
    profit,
    roi,
    CASE
        WHEN roi >= 3 THEN 'HIT'
        WHEN roi >= 1 THEN 'SUCCESS'
        ELSE 'FLOP'
    END AS category
FROM {catalogo}.golden.movie_master
"""
spark.sql(query2)

### features para ML

In [0]:
query3 = f"""

CREATE OR REPLACE VIEW {catalogo}.golden.view_sl_recommendation_features AS
SELECT
    id,
    title,
    genres,
    user_score,
    vote_count,
    
    -- Popularidad derivada
    (user_score * LOG10(vote_count + 1)) AS popularity_score,
    
    -- Bucket de duración
    CASE
        WHEN runtime_min < 90 THEN 'short'
        WHEN runtime_min BETWEEN 90 AND 120 THEN 'medium'
        ELSE 'long'
    END AS runtime_bucket,
    
    -- Año como feature
    release_year,
    
    -- ROI como feature
    roi
FROM {catalogo}.golden.movie_master

"""
spark.sql(query3)

### top para usuarios finales

In [0]:
query4 = f"""

CREATE OR REPLACE VIEW {catalogo}.golden.view_sl_top_content AS
SELECT *
FROM (
    SELECT
        id,
        title,
        user_score,
        revenue_usd,
        roi,
        ROW_NUMBER() OVER (ORDER BY user_score DESC) AS rank_rating,
        ROW_NUMBER() OVER (ORDER BY revenue_usd DESC) AS rank_revenue,
        ROW_NUMBER() OVER (ORDER BY roi DESC) AS rank_roi
    FROM {catalogo}.golden.movie_master
) t
WHERE rank_rating <= 10
   OR rank_revenue <= 10
   OR rank_roi <= 10

"""
spark.sql(query4)

In [0]:
%sql
select * from catalog_movie_project.golden.view_sl_kpi_genre


In [0]:
%sql
select * from catalog_movie_project.golden.view_sl_top_content